# Entrenamiento — Modelo custom: CNN-stem + Transformer (variante: 6 clases, con exclusión de calidad)

Décimo y último modelo del roster, el único que no sale directo de `timm`. Combina explícitamente dos partes de origen distinto:

1. **CNN stem preentrenado**: `resnet18` de ImageNet, truncado después de `layer3` (se descarta `layer4` y la cabeza original) — salida 14×14×256. Es el mismo punto de corte que usa el "Hybrid ViT" del paper original de ViT (Dosovitskiy et al., 2020) para combinar CNN+Transformer.
2. **Encoder Transformer desde cero**: proyección 1×1 a `d_model=256`, 196 tokens (14×14) + 1 token CLS aprendido, embedding posicional aprendido, 4 capas de atención (8 cabezas, MLP 4×, GELU, pre-norm), cabeza de clasificación sobre el token CLS final.

A diferencia de CoAtNet (etapas completas conv→atención) y MaxViT (conv + doble atención en cada bloque), aquí la CNN y el Transformer son dos bloques separados y secuenciales — la CNN hace toda la extracción local y solo entrega tokens ya reducidos en resolución, el Transformer aporta el modelado global. Al ser ensamblado específicamente para este proyecto (no una arquitectura publicada con nombre propio), es el aporte propio de la comparación, no "otro modelo más".

**Diferencia clave en el entrenamiento en 2 fases** respecto a los otros 9 modelos: ahí la fase 1 solo entrena una cabeza lineal ya bien inicializada por el preentrenamiento. Aquí, en fase 1, se entrena desde cero el bloque Transformer COMPLETO (proyección, token CLS, embedding posicional, encoder, cabeza) mientras el backbone CNN permanece congelado — por eso `get_classifier()` en este modelo devuelve todo ese bloque, no solo la última capa. Por la misma razón se le dan más épocas a la fase 1 (`PHASE1_EPOCHS = 12` en vez de 8) — hay bastante más que aprender desde cero que en los demás modelos.

**Antes de correr en Colab**: subir a la carpeta `BioMedInformatics` de Drive tanto `dataset_final.zip` (comprimir la carpeta `dataset_final/` completa) como `entrenamiento_utils.py` (este mismo directorio, `notebooks_entrenamiento/entrenamiento_utils.py`).

## 0. Configuración

In [ ]:
import sys

RUNNING_IN_COLAB = "google.colab" in sys.modules
SMOKE_TEST = False  # True solo para una corrida de prueba rápida (pocas imágenes, 1+1 épocas)

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/BioMedInformatics"  # ajustar si la carpeta se llama distinto

if RUNNING_IN_COLAB:
    !pip install -q timm
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/dataset_final
    !unzip -q -n {DRIVE_PROJECT_DIR}/dataset_final.zip -d /content/
    !cp {DRIVE_PROJECT_DIR}/entrenamiento_utils.py /content/
    sys.path.insert(0, "/content")
else:
    sys.path.insert(0, ".")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torchvision
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

import entrenamiento_utils as tu

MODEL_NAME = "custom_cnn_transformer"
VARIANT_NAME = "6clases_con_exclusion"
CLASSES = tu.CLASSES_6

DATA_ROOT = Path("/content/dataset_final") if RUNNING_IN_COLAB else Path("../dataset_final")
MANIFEST_PATH = DATA_ROOT / "manifiestos" / f"manifiesto_{VARIANT_NAME}.csv"

# Los resultados van a Drive (persisten entre sesiones) cuando se corre en Colab -- DATA_ROOT
# sí puede vivir en el disco local porque el dataset se puede volver a descomprimir gratis,
# pero horas de entrenamiento no se pueden regenerar gratis si se pierden al cerrar la sesión.
RESULTS_DIR = Path(DRIVE_PROJECT_DIR) / "resultados_entrenamiento" if RUNNING_IN_COLAB else Path("../resultados/entrenamiento")
CHECKPOINT_PATH = RESULTS_DIR / f"{MODEL_NAME}__{VARIANT_NAME}_best.pt"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True  # entradas de tamaño fijo -> cuDNN elige el algoritmo más rápido

# Desactiva los kernels de atención "flash"/eficiente-en-memoria de PyTorch y deja solo el kernel
# clásico ("math", más lento pero sin este problema). Se agregó tras un error real entrenando
# MaxViT en Colab (RuntimeError: "LSE is not correctly aligned (strideH)") -- un bug conocido del
# kernel flash-attention con ciertas formas de tensor, no algo de nuestros datos ni de este código.
# Con las secuencias cortas de este proyecto (~200-260 tokens) el costo de rendimiento es mínimo.
if torch.cuda.is_available():
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

print(f"Dispositivo: {device} | RUNNING_IN_COLAB={RUNNING_IN_COLAB} | SMOKE_TEST={SMOKE_TEST}")
print(f"Resultados en: {RESULTS_DIR}")

## 1. Cargar el manifiesto de la variante

In [ ]:
splits = tu.load_manifest(MANIFEST_PATH, CLASSES)

if SMOKE_TEST:
    splits = {
        name: df.groupby("target_class").head(4).reset_index(drop=True)
        for name, df in splits.items()
    }

for name, df in splits.items():
    print(f"{name}: {len(df)} imágenes")
splits["train"]["target_class"].value_counts()

## 2. Crear el modelo y las transformaciones de imagen
Este modelo no sale de `timm`, así que se define aquí mismo (clases `PosEmbed` y `CNNTransformerHybrid`). El tamaño de entrada (224×224) y la normalización (media/desviación estándar de ImageNet) se fijan a mano porque el backbone (`resnet18`) se preentrenó con esos valores — a diferencia de los otros 9 modelos, aquí no hay `timm.data.resolve_data_config` que los infiera solo.

In [ ]:
D_MODEL = 256
N_LAYERS = 4
N_HEADS = 8
MLP_RATIO = 4
DROPOUT = 0.1
IMG_SIZE = 224


class PosEmbed(nn.Module):
    """Token CLS + embedding posicional aprendido. Van en un submódulo propio (no como
    `nn.Parameter` sueltos en el modelo principal) para que `get_classifier()` los incluya
    correctamente como parte de la cabeza nueva -- un ModuleList solo recoge parámetros de
    los MÓDULOS que contiene, no de Parameters sueltos guardados en otro lado."""

    def __init__(self, n_tokens: int, d_model: int):
        super().__init__()
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_tokens, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        cls = self.cls_token.expand(tokens.shape[0], -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        return tokens + self.pos_embed


class CNNTransformerHybrid(nn.Module):
    """CNN stem (ResNet18 preentrenado, truncado en layer3) + encoder Transformer (desde cero)."""

    def __init__(self, num_classes: int, img_size: int = IMG_SIZE, d_model: int = D_MODEL,
                 n_layers: int = N_LAYERS, n_heads: int = N_HEADS, mlp_ratio: int = MLP_RATIO,
                 dropout: float = DROPOUT, pretrained: bool = True):
        super().__init__()
        weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        resnet = torchvision.models.resnet18(weights=weights)

        # Parte nueva (entrenada desde cero) registrada PRIMERO a propósito: `unfreeze_last_fraction`
        # (en entrenamiento_utils.py) recorre named_parameters() EN ORDEN y descongela la última
        # fracción en fase 2 -- si el backbone se registrara primero, esa "última fracción" caería
        # dentro de esta parte (ya entrenable desde fase 1) y el backbone nunca se afinaría en fase 2.
        # Registrándolo al final, la última fracción cae en las capas más profundas del backbone
        # (layer3), que es la intención real de la fase 2 (igual que en los otros 9 modelos).
        self.proj = nn.Conv2d(256, d_model, kernel_size=1)  # 256 = canales de salida de resnet18.layer3
        n_tokens_side = img_size // 16  # stride total hasta layer3: 2(conv1) x 2(maxpool) x 2(layer2) x 2(layer3)
        self.embed = PosEmbed(n_tokens_side * n_tokens_side + 1, d_model)  # +1 por el token CLS
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * mlp_ratio,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

        # Backbone CNN preentrenado (ImageNet), truncado después de layer3 -- registrado AL FINAL,
        # ver nota arriba.
        self.backbone = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(x)                    # (B, 256, 14, 14)
        feat = self.proj(feat)                      # (B, d_model, 14, 14)
        tokens = feat.flatten(2).transpose(1, 2)     # (B, 196, d_model)
        tokens = self.embed(tokens)                  # (B, 197, d_model) -- agrega CLS + posicional
        tokens = self.encoder(tokens)
        cls_out = self.norm(tokens[:, 0])
        return self.head(cls_out)

    def get_classifier(self) -> nn.Module:
        """Todo lo que NO es el backbone preentrenado -- lo que `train_two_phase` entrena desde
        la fase 1 (backbone congelado). Un ModuleList, no una lista de Parameters, para que
        incluya correctamente los de `embed` (cls_token/pos_embed)."""
        return nn.ModuleList([self.proj, self.embed, self.encoder, self.norm, self.head])


model = CNNTransformerHybrid(num_classes=len(CLASSES), pretrained=True)
input_size = (IMG_SIZE, IMG_SIZE)
# Estándar de ImageNet -- el backbone (resnet18) se preentrenó con esta normalización.
data_config = {"mean": (0.485, 0.456, 0.406), "std": (0.229, 0.224, 0.225)}
print("Tamaño de entrada:", input_size, "| normalización:", data_config["mean"], data_config["std"])

n_total = sum(p.numel() for p in model.parameters())
n_backbone = sum(p.numel() for p in model.backbone.parameters())
n_transformer = sum(p.numel() for p in model.get_classifier().parameters())
print(f"Parámetros totales: {n_total:,} (backbone CNN: {n_backbone:,} | bloque Transformer: {n_transformer:,})")

In [ ]:
# Aumento de datos (solo train) — mismo esquema que en el artículo V2: flip horizontal, rotación
# leve, zoom leve, ajuste de contraste leve, para no distorsionar patrones clínicos relevantes.
train_transform = transforms.Compose([
    transforms.Resize(input_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=15, scale=(0.9, 1.1)),
    transforms.ColorJitter(contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=data_config["mean"], std=data_config["std"]),
])
eval_transform = transforms.Compose([
    transforms.Resize(input_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=data_config["mean"], std=data_config["std"]),
])

## 3. Balanceo de clases: muestreo ponderado (no pesos de pérdida)

Silicosis (737 imágenes de train) y STB (529) tienen mucha menos data que las clases grandes (~4,200 cada una) — con solo pesos de pérdida, el modelo sigue *viendo* pocas imágenes de esas clases por época, solo se le da más peso al error cuando se equivoca en ellas. Un `WeightedRandomSampler` ataca el problema distinto: hace que el propio DataLoader muestree las clases raras con más frecuencia dentro de cada época (con reemplazo), así el modelo ve una mezcla más pareja de las 6 clases en cada batch.

**No se usan los dos mecanismos a la vez** (muestreo balanceado + pesos de pérdida) porque se sobre-compensaría el desbalance — se elige uno. Aquí se usa el sampler; la pérdida queda sin ponderar (`class_weights=None` en la Sección 6).

In [ ]:
conteo_train = splits["train"]["target_class"].value_counts()
print("Distribución real en train (antes de balancear el muestreo):")
print(conteo_train)

train_sampler = tu.make_weighted_sampler(splits["train"], CLASSES)

## 4. Datasets y DataLoaders

In [ ]:
BATCH_SIZE = 8 if SMOKE_TEST else 32
NUM_WORKERS = 0 if (RUNNING_IN_COLAB is False or SMOKE_TEST) else 2

train_ds = tu.CXRDataset(splits["train"], DATA_ROOT, CLASSES, transform=train_transform)
val_ds = tu.CXRDataset(splits["val"], DATA_ROOT, CLASSES, transform=eval_transform)
test_ds = tu.CXRDataset(splits["test"], DATA_ROOT, CLASSES, transform=eval_transform)

# pin_memory acelera la transferencia CPU->GPU; persistent_workers evita relanzar los
# workers en cada época (con NUM_WORKERS=0, como en local, ninguno de los dos hace nada)
loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                     persistent_workers=NUM_WORKERS > 0)

# shuffle=False porque el orden ya lo decide el sampler (son mutuamente excluyentes en PyTorch)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

# Verificación rápida: un batch real debe cargar sin error y tener la forma esperada
images, labels = next(iter(train_loader))
print("Forma de un batch:", images.shape, labels.shape)

In [ ]:
# Confirmar que el sampler de verdad está balanceando: contar clases en varios batches muestreados
from collections import Counter
conteo_muestreado = Counter()
for i, (_, labels) in enumerate(train_loader):
    conteo_muestreado.update(labels.tolist())
    if i >= 20:
        break
print("Distribución observada en ~20 batches muestreados (debería verse mucho más pareja que el conteo real):")
for i, c in enumerate(CLASSES):
    print(f"  {c}: {conteo_muestreado[i]}")

## 5. Entrenar en 2 fases

In [ ]:
# Más épocas en fase 1 que en los otros 9 modelos: aquí no solo se ajusta una cabeza ya bien
# inicializada por el preentrenamiento, se entrena el bloque Transformer COMPLETO desde cero.
PHASE1_EPOCHS = 1 if SMOKE_TEST else 12
PHASE2_EPOCHS = 1 if SMOKE_TEST else 12

model, history = tu.train_two_phase(
    model, train_loader, val_loader, class_weights=None, device=device,
    phase1_epochs=PHASE1_EPOCHS, phase1_lr=1e-3,
    phase2_epochs=PHASE2_EPOCHS, phase2_lr=1e-5,
    unfreeze_fraction=0.3, patience=5,
    checkpoint_path=CHECKPOINT_PATH,
)

In [ ]:
import pandas as pd
tu.plot_training_curves(history, MODEL_NAME, VARIANT_NAME, RESULTS_DIR)
plt.show()

## 6. Evaluar sobre el conjunto de test

In [ ]:
metrics = tu.evaluate_model(model, test_loader, CLASSES, device)
print(f"Accuracy: {metrics['accuracy']:.3f}")
print(f"Precision macro: {metrics['precision_macro']:.3f}")
print(f"Recall macro: {metrics['recall_macro']:.3f}")
print(f"F1 macro: {metrics['f1_macro']:.3f}")
print(f"AUC-ROC macro: {metrics['auc_roc_macro']:.3f}")
print(f"Tiempo de inferencia: {metrics['inference_time_ms_per_image']:.2f} ms/imagen")

In [ ]:
pd.DataFrame(metrics["per_class"])

In [ ]:
tu.plot_confusion_matrix(metrics, CLASSES, MODEL_NAME, VARIANT_NAME, RESULTS_DIR)
plt.show()

### 6.1 Curvas ROC por clase
Uno-contra-todos, con el AUC ya calculado en `evaluate_model`.

In [ ]:
tu.plot_roc_curves(metrics, CLASSES, MODEL_NAME, VARIANT_NAME, RESULTS_DIR)
plt.show()

### 6.2 Curvas Precision-Recall
Más informativa que ROC cuando el test está desbalanceado entre clases (Silicosis/STB tienen mucho menos soporte que Normal/Pneumonia/COVID-19) — ROC incluye los verdaderos negativos, que ahí dominan y pueden maquillar una precisión mala en la clase rara.

In [ ]:
tu.plot_pr_curves(metrics, CLASSES, MODEL_NAME, VARIANT_NAME, RESULTS_DIR)
plt.show()

### 6.3 Precision/Recall/F1 por clase
Mismos números de la tabla de la Sección 6, en forma de gráfico (barras de error = IC 95% de Wilson en precision/recall) — más fácil de comparar entre clases de un vistazo.

In [ ]:
tu.plot_per_class_metrics(metrics, MODEL_NAME, VARIANT_NAME, RESULTS_DIR)
plt.show()

## 7. Costo computacional

In [ ]:
params_info = tu.count_params(model)
size_mb = tu.model_size_mb(model)
print(f"Parámetros totales: {params_info['total_params']:,}")
print(f"Tamaño en disco: {size_mb:.2f} MB")
print(f"Tiempo de inferencia: {metrics['inference_time_ms_per_image']:.2f} ms/imagen")

## 8. Guardar resultados
Mismo formato para los 10 modelos, para el notebook de comparación final.

In [ ]:
out_path = tu.save_results(MODEL_NAME, VARIANT_NAME, history, metrics, params_info, size_mb, RESULTS_DIR)
print(f"Resultados guardados en: {out_path}")
print(f"Métricas por clase (CSV): {RESULTS_DIR / f'{MODEL_NAME}__{VARIANT_NAME}_metricas_por_clase.csv'}")
print(f"Predicciones crudas (y_true/y_pred/y_prob, para pruebas de significancia u otras figuras "
      f"más adelante): {RESULTS_DIR / f'{MODEL_NAME}__{VARIANT_NAME}_predicciones.npz'}")
print(f"Tabla comparativa entre TODOS los modelos: {RESULTS_DIR / 'comparacion_modelos.csv'}")
print(f"Checkpoint del mejor modelo: {CHECKPOINT_PATH}")
print(f"Tiempo total de entrenamiento: {sum(history['epoch_time_s']) / 60:.1f} min")

## Notas
- `SMOKE_TEST = True` al inicio permite verificar que todo el pipeline corre de punta a punta (pocas imágenes, 1 época por fase) antes de lanzar el entrenamiento completo — útil para no gastar horas de GPU si hay un error de código.
- El balanceo de clases se hace con `WeightedRandomSampler` (Sección 3), no con pesos en la pérdida — por eso `train_two_phase` se llama con `class_weights=None`.
- Los resultados quedan en `resultados_entrenamiento/` dentro de la carpeta `BioMedInformatics` de Drive (persisten entre sesiones de Colab, no se pierden si la sesión se desconecta), listos para que el notebook de comparación final los junte con los de los otros 9 modelos.